# Phase 4: Data Merge

This notebook merges cleaned transactions with the processed property GIS layer.

Scope:
- load cleaned transactions and processed property GIS
- normalize merge keys
- aggregate duplicate property keys
- create a merged transaction-property base dataset


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import geopandas as gpd
import pandas as pd

from backend.src.config import configure_logging, ensure_directories, settings
from backend.src.data_merge import merge_transactions_with_property, save_merged_dataset

configure_logging()
ensure_directories()
PROJECT_ROOT, settings


(WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc'),
 Settings(project_root=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc'), raw_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data'), interim_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/interim'), processed_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/processed'), reports_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports'), models_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/models'), transaction_file='tran_data.xlsx', property_shapefile='ai_usecase_data240326.shp', road_shapefile='ai_usecase_road240326.shp', facility_shapefile='ai_usecase_facilities240326.shp', log_level='INFO'))

In [2]:
transactions = pd.read_parquet(settings.interim_data_dir / 'cleaned_transactions.parquet')
property_gdf = gpd.read_parquet(settings.interim_data_dir / 'property_gis.parquet')
transactions.shape, property_gdf.shape


((273587, 56), (111497, 29))

In [3]:
merged_gdf, summary = merge_transactions_with_property(transactions, property_gdf)
summary


MergeSummary(transaction_input_rows=273587, property_input_rows=111497, property_unique_merge_keys=93775, property_duplicate_merge_key_rows=25532, matched_transaction_rows=207442, unmatched_transaction_rows=66145, match_coverage_percent=75.82, output_rows=273587, merge_key_columns=['district_name_norm', 'ps_code_norm', 'mouza_code_norm', 'plot_no_norm'])

In [4]:
merged_gdf.shape


(273587, 79)

In [5]:
merged_gdf[['property_district_Name', 'ps_code', 'mouza_code', 'plot_no', 'property_match_found', 'property_record_count']].head()


,property_district_Name,ps_code,mouza_code,plot_no,property_match_found,property_record_count
0,Paschim Bardhaman,19,91,534,True,1.0
1,Paschim Bardhaman,19,58,1477,True,1.0
2,Paschim Bardhaman,20,111,264,True,1.0
3,Paschim Bardhaman,19,91,953,True,1.0
4,Paschim Bardhaman,20,107,5542,True,1.0


In [6]:
merged_path, summary_path = save_merged_dataset(
    merged_gdf,
    summary,
    settings.interim_data_dir,
    settings.reports_dir,
)
merged_path, summary_path


2026-04-28 15:30:40,102 | INFO | src.data_merge | Saved merged transaction-property dataset to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\interim\transactions_property_merged.parquet
2026-04-28 15:30:40,103 | INFO | src.data_merge | Saved data merge summary to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\reports\phase_4_data_merge_summary.json


(WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/interim/transactions_property_merged.parquet'),
 WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports/phase_4_data_merge_summary.json'))

In [7]:
json.loads(Path(summary_path).read_text())


{'transaction_input_rows': 273587,
 'property_input_rows': 111497,
 'property_unique_merge_keys': 93775,
 'property_duplicate_merge_key_rows': 25532,
 'matched_transaction_rows': 207442,
 'unmatched_transaction_rows': 66145,
 'match_coverage_percent': 75.82,
 'output_rows': 273587,
 'merge_key_columns': ['district_name_norm',
  'ps_code_norm',
  'mouza_code_norm',
  'plot_no_norm']}